# 风险字段语义与价格口径
固定检查已有输入，不新增预测信息、不训练、不按收益筛选字段。区分数量纲差异、真实不同概念以及同概念不同价格源。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.data.qlib_native import QlibProviderConfig,load_qlib_feature_frame
from src.workflow.minute_action_value import _source
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/risk_feature_semantics_20261014';plan=json.loads((card/'plan.json').read_text());qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us')
source,audit=_source(root,plan['source'],['inputs/features','inputs/common']);a=source['inputs/features'];b=source['inputs/common'].reindex(a.index)
planes={};manifests={}
for key,relative in plan['providers'].items():
 p=root/relative;manifest=p/'qlib_store_manifest.json';assert hashlib.sha256(manifest.read_bytes()).hexdigest()==plan['provider_manifest_sha256'][key]
 manifests[key]=json.loads(manifest.read_text());planes[key]=load_qlib_feature_frame(QlibProviderConfig(p),instruments=['SOXX'],fields=['$close'],start='2010-03-11',end=plan['as_of']).xs('SOXX',level='instrument')['$close'].astype(float)
calc={}
for key,close in planes.items():
 ret=close.pct_change(fill_method=None)
 calc[key+'_vol20_sample']=ret.rolling(20,min_periods=20).std(ddof=1).reindex(a.index)
 calc[key+'_vol20_population']=ret.rolling(20,min_periods=20).std(ddof=0).reindex(a.index)
 calc[key+'_drawdown60']= (close/close.rolling(60,min_periods=60).max()-1).reindex(a.index)
comparison=pd.DataFrame({'primary_vol20':a.soxx_vol20,'common_vol20_as_sample':b.vol20*np.sqrt(20/19),'primary_drawdown60':a.soxx_drawdown60,'common_drawdown60':b.drawdown60,**calc})
comparison['cross_vol_difference']=comparison.primary_vol20-comparison.common_vol20_as_sample
comparison['cross_drawdown_difference']=comparison.primary_drawdown60-comparison.common_drawdown60
pairs=[('primary_vol20','execution_vol20_sample'),('common_vol20_as_sample','daily_vol20_sample'),('primary_drawdown60','execution_drawdown60'),('common_drawdown60','daily_drawdown60')]
reconstruction=[]
for frozen,current in pairs:
 delta=(comparison[frozen]-comparison[current]).abs();valid=delta.notna()
 reconstruction.append({'frozen_feature':frozen,'reconstructed_from':current,'observations':int(valid.sum()),'max_absolute_error':float(delta.max()),'median_absolute_error':float(delta.median()),'within_float_tolerance_rows':int(delta.le(1e-6).sum())})
annual=comparison.groupby(comparison.index.year).agg(sessions=('primary_vol20','size'),max_vol_difference=('cross_vol_difference',lambda x:x.abs().max()),max_drawdown_difference=('cross_drawdown_difference',lambda x:x.abs().max()))
anchor=pd.DataFrame({'field':['primary:rv20','primary:rv60','primary:soxx_vol20','common:vol20','primary:soxx_drawdown60','common:drawdown60','primary:dist_high_252'],'value':[a.loc[plan['anchor'],'rv20'],a.loc[plan['anchor'],'rv60'],a.loc[plan['anchor'],'soxx_vol20'],b.loc[plan['anchor'],'vol20'],a.loc[plan['anchor'],'soxx_drawdown60'],b.loc[plan['anchor'],'drawdown60'],a.loc[plan['anchor'],'dist_high_252']]})
semantics=pd.DataFrame([
 {'field':'primary:rv20','asset':'SOXL','quantity':'annualized close-return std','window':20,'min_periods':20,'ddof':1,'scale':'sqrt252','source':'daily'},
 {'field':'primary:rv60','asset':'SOXL','quantity':'annualized close-return std','window':60,'min_periods':30,'ddof':1,'scale':'sqrt252','source':'daily'},
 {'field':'primary:soxx_vol20','asset':'SOXX','quantity':'daily close-return std','window':20,'min_periods':20,'ddof':1,'scale':'1','source':'execution'},
 {'field':'common:vol20','asset':'SOXX','quantity':'daily close-return std','window':20,'min_periods':10,'ddof':0,'scale':'1','source':'daily'},
 {'field':'primary:soxx_drawdown60','asset':'SOXX','quantity':'distance from rolling close maximum','window':60,'min_periods':60,'ddof':None,'scale':'1','source':'execution'},
 {'field':'common:drawdown60','asset':'SOXX','quantity':'distance from rolling close maximum','window':60,'min_periods':20,'ddof':None,'scale':'1','source':'daily'},
 {'field':'primary:dist_high_252','asset':'SOXL','quantity':'distance from rolling close maximum','window':252,'min_periods':100,'ddof':None,'scale':'1','source':'daily'}])
summary={'experiment':card.name,'decision':'risk_units_distinguished_cross_price_plane_inconsistency_measured','new_fits':0,'new_backtests':0,'parameter_searches':0,'outcome_correlations':0,'objective_achieved':False,'rows':len(comparison),'maximum_cross_plane_vol_difference':float(comparison.cross_vol_difference.abs().max()),'maximum_cross_plane_drawdown_difference':float(comparison.cross_drawdown_difference.abs().max()),'provider_uri':str(root/plan['providers']['daily']),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/data.html','current_price_reconstruction_is_historical_vintage_certification':False,'production_changed':False,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily','sources/*']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='risk_semantics_no_fit',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0)
 rec.save_objects(artifact_path='inputs',plan=plan,source_audit=audit,current_provider_manifests=manifests)
 rec.save_objects(artifact_path='analysis',daily=comparison,annual=annual,reconstruction=pd.DataFrame(reconstruction),anchor=anchor,semantics=semantics,summary=summary)
 rec.save_objects(artifact_path='sources',high_chop=(root/'src/factor/high_chop_ml.py').read_text(),common_tail=(root/'src/factor/soxx_pit_tail_factors.py').read_text(),execution_features=(root/'src/workflow/soxs_defensive_hurdle.py').read_text())
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json')
for name,table in [('semantics',semantics),('reconstruction',pd.DataFrame(reconstruction)),('anchor',anchor)]:table.to_csv(card/(name+'.csv'),index=False)
annual.to_csv(card/'annual_source_difference.csv')
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':summary['provider_uri'],'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
for key,relative in plan['providers'].items():assert hashlib.sha256((root/relative/'qlib_store_manifest.json').read_bytes()).hexdigest()==plan['provider_manifest_sha256'][key]
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256']
print(pd.DataFrame(reconstruction).to_string(index=False));print(anchor.to_string(index=False))


In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import torch,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.model.minute_pit_allocator import _scaled
from src.workflow.pit_constituent_auxiliary_training_allocation import PitAuxiliaryAllocationNet,_project_legal_weights
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';torch.set_num_threads(1);card=root/'research/risk_feature_semantics_20261014';plan=json.loads((card/'plan.json').read_text());summary=json.loads((card/'summary.json').read_text());qlib.init(provider_uri=summary['provider_uri'],region='us')
s,_=_source(root,plan['source'],['inputs/features','inputs/common','inputs/frozen_parent_models','inputs/plan']);state=s['inputs/frozen_parent_models']['outer_refit'];spec=s['inputs/plan'];a=s['inputs/features'].loc['2023-01-03':'2026-10-08'].copy();b=s['inputs/common'].reindex(a.index).copy()
net=PitAuxiliaryAllocationNet(primary_feature_count=len(state['primary_features']),common_feature_count=len(state['common_features']),hidden_units=spec['hidden_units'],tail_hidden_units=spec['tail_hidden_units']);net.load_state_dict({k:torch.tensor(v) for k,v in state['model_state'].items()});net.eval()
def weights(primary,common):
 with torch.no_grad():
  logits,_,_=net(_scaled(primary,state['primary_features'],state['primary_scaler']),_scaled(common,state['common_features'],state['common_scaler']))
  return _project_legal_weights(logits,maximum_soxl_weight=spec['maximum_soxl_weight'],minimum_cash_weight=spec['minimum_cash_weight']).numpy()
baseline=weights(a,b);same_daily=a.copy();same_daily['soxx_vol20']=b.vol20*np.sqrt(20/19);same_daily['soxx_drawdown60']=b.drawdown60
same_execution=b.copy();same_execution['vol20']=a.soxx_vol20/np.sqrt(20/19);same_execution['drawdown60']=a.soxx_drawdown60
cases={'align_primary_to_daily':weights(same_daily,b),'align_common_to_execution':weights(a,same_execution)};rows=[]
for name,value in cases.items():
 diff=(value[:,2]-baseline[:,2])*100;anchor=a.index.get_loc(pd.Timestamp(plan['anchor']))
 rows.append({'case':name,'sessions':len(a),'mean_absolute_cash_change_pp':float(np.abs(diff).mean()),'maximum_absolute_cash_change_pp':float(np.abs(diff).max()),'anchor_base_cash_pct':float(baseline[anchor,2]*100),'anchor_case_cash_pct':float(value[anchor,2]*100),'anchor_cash_change_pp':float(diff[anchor])})
summary['symmetric_source_sensitivity']=rows;summary['sensitivity_role']='Frozen input interventions only; no refit, return evaluation, preferred source selection or executable target'
with R.uri_context(summary['recorder_uri']):
 rec=R.get_recorder(recorder_id=summary['recorder_id'],experiment_name=card.name)
 rec.save_objects(artifact_path='analysis',source_sensitivity=pd.DataFrame(rows),sensitivity_weights={'dates':a.index,'baseline':baseline,**cases},summary=summary)
 artifacts=Path(summary['recorder_uri'])/summary['experiment_id']/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},indent=2)+'\n')
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=summary['experiment_id']);(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');pd.DataFrame(rows).to_csv(card/'source_sensitivity.csv',index=False)
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256']
print(pd.DataFrame(rows).to_string(index=False))


# 风险字段核查：不同概念不能去重，来源差异不足以解释高现金

0拟合、0回测、0参数搜索、0未来收益相关检验。本卡读取冻结输入和两个原生Qlib价格库，分别重算风险量；随后对同一冻结网络做两个对称的输入口径干预，不比较收益、不选择“更好”来源。

明确字段含义：rv20/rv60是SOXL年化波动率，soxx_vol20是SOXX日波动率；common vol20也是SOXX日波动率但ddof=0。20个有效收益时，以sqrt(20/19)对齐样本标准差。dist_high_252是SOXL距252日最高收盘的距离，不能与SOXX的60日回撤直接合并。common和primary的60日回撤都基于收盘最大值，成熟窗口时公式相同，但价格来源不同。最低观测期的差异在semantics.csv单列。

3966个冻结样本，按各自来源重算，四项风险量全部在1e-6容差内匹配，最大误差约1.96e-8。对齐ddof后，两个价格源的日波动率最大差0.00052343，60日回撤最大差0.00800856（0.800856个百分点）；因此不是简单年度/日度换算错误。历史来源差异真实存在，但不能由当前重算认证历史首收或版本。

2023-11-03：SOXL rv20=0.84160（年化84.16%）；SOXX样本日波动率0.017804（1.7804%）；主分支SOXX60日回撤-6.8452%，公共分支-7.1758%；SOXL252日距高点-35.5766%。这几项表达不同资产、不同窗口与不同数量，不是同一个风险程度。

对2023-01-03至2026-10-08全部945个可推理信号，固定模型、原标准化和所有其他输入不变：将主分支两项风险对齐daily来源，现金目标平均绝对变化0.01238个百分点、最大0.35494个百分点；将公共分支对齐execution来源，平均0.00438个百分点、最大0.15734个百分点。锚点现金由75.1029%分别变为75.1356%和75.1275%。这945个输入日不等于已有完整账户的944个可比持有区间；本卡没有账户结算。

结论：已辨明风险特征语义及价格来源差异，但这两项来源差异不是该锚点高现金的主要解释。不要因为字段高相关就删除，也不要把口径修整冒充alpha。纯单位的正比例变换若同步变换训练均值和标准差，并不会改变标准化输入；本模型已逐字段标准化，不能把量纲差异本身宣称为已证实缺陷。

决定：risk_units_distinguished_cross_price_plane_inconsistency_measured。当前冻结模型和生产release未修改，稳定净超额未实现。停止继续围绕这两个口径寻找更好现金目标；后续应检查信息到配置决策的目标与约束关系，或有独立来源的新信息。保持当前不拟合要求。

[字段语义](semantics.csv) · [逐源重算](reconstruction.csv) · [固定模型敏感性](source_sensitivity.csv) · [年度来源差异](annual_source_difference.csv)

Recorder `8a20541d39f74f34be16e7cc7f6e38c8`。
